In [1]:
# If anything is missing, run (uncomment):
# !pip install torch torchvision torchaudio scikit-learn matplotlib pillow

import os, sys, time, math, csv, contextlib, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Union

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler

from PIL import Image
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, classification_report

import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

print({
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
})
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


{'python': '3.13.6 (tags/v3.13.6:4e66535, Aug  6 2025, 14:36:00) [MSC v.1944 64 bit (AMD64)]', 'torch': '2.8.0+cpu', 'torchvision': '0.23.0+cpu'}
device: cpu


In [2]:
@dataclass
class Config:
    # Paths
    data_dir: str = r"E:\1 Paper MCT\Cutting Tool Paper\Dataset\cutting tool data\test_data_40_images"
    out_dir: str = "./runs/fsl_fault_diag_notebook"

    # Data
    image_size: int = 224
    normalize_mean = (0.485, 0.456, 0.406)
    normalize_std  = (0.229, 0.224, 0.225)

    # Few-shot episodic settings
    n_way: int = 5
    k_shot: int = 5
    q_queries: int = 10
    episodes_per_epoch: int = 60   # increase later if you want
    val_episodes: int = 30
    test_episodes: int = 60

    # Training
    max_epochs: int = 5            # increase to ~30 for final accuracy
    lr: float = 3e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0

    # Backbone / head
    embed_dim: int = 256
    use_pretrained: bool = False   # keep False to avoid downloading weights
    cov_shrinkage: float = 0.1
    temperature: float = 1.0

    # 🔬 Research toggles (novelties & ablations)
    use_attention: bool = True       # lightweight Self-Attention on CNN maps
    use_reliability: bool = True     # reliability-weighted prototypes
    metric: str = "maha"             # "maha" (Cholesky) or "euclid"

    # Repro / device
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


Config(data_dir='E:\\1 Paper MCT\\Cutting Tool Paper\\Dataset\\cutting tool data\\test_data_40_images', out_dir='./runs/fsl_fault_diag_notebook', image_size=224, n_way=5, k_shot=5, q_queries=10, episodes_per_epoch=60, val_episodes=30, test_episodes=60, max_epochs=5, lr=0.0003, weight_decay=0.0001, grad_clip=1.0, embed_dim=256, use_pretrained=False, cov_shrinkage=0.1, temperature=1.0, use_attention=True, use_reliability=True, metric='maha', seed=42, device='cpu')


In [3]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def get_device(pref: str = "cuda"):
    if pref == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


In [4]:
from typing import Any

def discover_images(root_dir: str) -> Dict[str, List[str]]:
    class_to_paths: Dict[str, List[str]] = {}
    for cls in sorted(os.listdir(root_dir)):
        cdir = os.path.join(root_dir, cls)
        if not os.path.isdir(cdir):
            continue
        paths = []
        for fn in os.listdir(cdir):
            if os.path.splitext(fn)[1].lower() in IMG_EXTS:
                paths.append(os.path.join(cdir, fn))
        if paths:
            class_to_paths[cls] = sorted(paths)
    if not class_to_paths:
        raise RuntimeError(f"No classes with images found in: {root_dir}")
    return class_to_paths

def preload_images(class_to_paths: Dict[str, List[str]]) -> Dict[str, List[Image.Image]]:
    cached: Dict[str, List[Image.Image]] = {}
    total = 0
    for cls, paths in class_to_paths.items():
        imgs = [Image.open(p).convert("RGB") for p in paths]
        cached[cls] = imgs
        total += len(imgs)
    print(f"[data] RAM-cached {total} images across {len(cached)} classes.")
    return cached

def default_transforms(cfg: Config, split: str):
    if split == "train":
        aug = [
            transforms.Resize((cfg.image_size, cfg.image_size)),
            transforms.RandomApply([transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02)], p=0.7),
            transforms.RandomHorizontalFlip(),
            transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
            transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(cfg.normalize_mean, cfg.normalize_std),
            transforms.RandomErasing(p=0.25, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0),
        ]
    else:
        aug = [
            transforms.Resize((cfg.image_size, cfg.image_size)),
            transforms.ToTensor(),
            transforms.Normalize(cfg.normalize_mean, cfg.normalize_std),
        ]
    return transforms.Compose(aug)

def sample_episode(
    data_dict: Dict[str, List[Union[str, Image.Image]]],
    n_way: int, k_shot: int, q_queries: int, split: str, cfg: Config
):
    assert n_way <= len(data_dict), "n_way exceeds number of classes."
    classes = random.sample(list(data_dict.keys()), n_way)

    supp_imgs, supp_labels, qry_imgs, qry_labels = [], [], [], []
    tform = default_transforms(cfg, "train" if split == "train" else "eval")

    for ci, cls in enumerate(classes):
        items = data_dict[cls]
        assert len(items) >= k_shot + q_queries, f"Not enough images in class {cls} for k_shot+q_queries"
        picks = random.sample(items, k_shot + q_queries)
        supp, qry = picks[:k_shot], picks[k_shot:]
        for obj in supp:
            img = Image.open(obj).convert("RGB") if isinstance(obj, str) else obj
            supp_imgs.append(tform(img)); supp_labels.append(ci)
        for obj in qry:
            img = Image.open(obj).convert("RGB") if isinstance(obj, str) else obj
            qry_imgs.append(tform(img));  qry_labels.append(ci)

    supp_x = torch.stack(supp_imgs, 0)
    qry_x  = torch.stack(qry_imgs,  0)
    supp_y = torch.tensor(supp_labels, dtype=torch.long)
    qry_y  = torch.tensor(qry_labels,  dtype=torch.long)
    return (supp_x, supp_y, qry_x, qry_y, classes)


In [5]:
class SelfAttention2D(nn.Module):
    def __init__(self, channels: int, heads: int = 4, dim_head: int = 32):
        super().__init__()
        inner = heads * dim_head
        self.heads = heads
        self.to_q = nn.Conv2d(channels, inner, 1, bias=False)
        self.to_k = nn.Conv2d(channels, inner, 1, bias=False)
        self.to_v = nn.Conv2d(channels, inner, 1, bias=False)
        self.proj = nn.Conv2d(inner, channels, 1, bias=False)
        self.scale = dim_head ** -0.5

    def forward(self, x):
        b, c, h, w = x.shape
        q = self.to_q(x).view(b, self.heads, -1, h*w)
        k = self.to_k(x).view(b, self.heads, -1, h*w)
        v = self.to_v(x).view(b, self.heads, -1, h*w)
        attn = torch.softmax((q.transpose(-2, -1) @ k) * self.scale, dim=-1)  # [B,H,HW,HW]
        out = (attn @ v.transpose(-2, -1)).transpose(-2, -1)                  # [B,H,Dh,HW]
        out = out.reshape(b, -1, h, w)
        return self.proj(out) + x

class QualityNet(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, d//2),
            nn.ReLU(inplace=True),
            nn.Linear(d//2, 1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

class ResNetBackbone(nn.Module):
    def __init__(self, out_dim: int = 256, use_pretrained: bool = False, use_attention: bool = True):
        super().__init__()
        w = ResNet18_Weights.DEFAULT if use_pretrained else None
        base = resnet18(weights=w)
        self.features = nn.Sequential(*list(base.children())[:-2])
        self.num_ch = base.fc.in_features  # 512 for resnet18
        self.attn = SelfAttention2D(self.num_ch, heads=4, dim_head=32) if use_attention else nn.Identity()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(self.num_ch, out_dim)
        self.quality = QualityNet(out_dim)

    def forward(self, x):
        fmap = self.features(x)
        fmap = self.attn(fmap)
        emb  = self.pool(fmap).flatten(1)
        emb  = self.proj(emb)
        return emb

    def quality_score(self, emb):
        return self.quality(emb)


In [6]:
def classwise_mean_cov(supp_emb: torch.Tensor, supp_y: torch.Tensor, n_way: int, shrink: float = 0.1):
    device = supp_emb.device
    n, d = supp_emb.shape
    means = torch.zeros(n_way, d, device=device)
    covs  = torch.zeros(n_way, d, d, device=device)
    eps = 1e-5
    eye = torch.eye(d, device=device)
    for c in range(n_way):
        x = supp_emb[supp_y == c]
        mu = x.mean(dim=0, keepdim=True)
        xm = x - mu
        cov = (xm.t() @ xm) / max(1, x.shape[0] - 1) + eps * eye
        cov = (1.0 - shrink) * cov + shrink * eye
        means[c] = mu
        covs[c]  = cov
    L = torch.linalg.cholesky(covs)  # [n_way,d,d]
    return means, L

class AdaptiveProtoHead(nn.Module):
    def __init__(self, temperature: float = 1.0, shrinkage: float = 0.1, metric: str = "maha"):
        super().__init__()
        assert metric in ("maha", "euclid")
        self.temperature = temperature
        self.shrinkage = shrinkage
        self.metric = metric

    @staticmethod
    def weighted_mean(emb: torch.Tensor, labels: torch.Tensor, n_way: int, w: torch.Tensor):
        device = emb.device
        n, d = emb.shape
        means = torch.zeros(n_way, d, device=device)
        for c in range(n_way):
            mask = (labels == c)
            wc = w[mask].clamp(min=1e-3).unsqueeze(-1)
            xc = emb[mask]
            means[c] = (wc * xc).sum(dim=0) / wc.sum(dim=0)
        return means

    def forward(self, supp_emb, supp_y, qry_emb, n_way, reliability):
        means_w = self.weighted_mean(supp_emb, supp_y, n_way, reliability)  # [n_way,D]

        if self.metric == "euclid":
            diff = qry_emb.unsqueeze(1) - means_w.unsqueeze(0)
            d2 = (diff ** 2).sum(dim=-1)
            logits = -d2 / self.temperature
            return logits, means_w

        # Mahalanobis via Cholesky (stable, no inverse)
        means, L = classwise_mean_cov(supp_emb, supp_y, n_way, self.shrinkage)
        diff = qry_emb.unsqueeze(1) - means_w.unsqueeze(0)  # [Nq,n_way,D]
        Nq, C, D = diff.shape
        diff2 = diff.permute(1,0,2).reshape(C, Nq, D)       # [n_way,Nq,D]
        y = torch.linalg.solve_triangular(L, diff2.transpose(1,2), upper=False)
        z = torch.linalg.solve_triangular(L.transpose(-1,-2), y, upper=True)
        d2 = (z**2).sum(dim=1).transpose(0,1)               # [Nq,n_way]
        logits = -d2 / self.temperature
        return logits, means_w


In [7]:
def xent_loss(logits: torch.Tensor, labels: torch.Tensor):
    return nn.functional.cross_entropy(logits, labels)

def proto_compactness_loss(emb: torch.Tensor, labels: torch.Tensor, means: torch.Tensor):
    diffs = emb - means[labels]
    return (diffs.pow(2).sum(dim=1)).mean()


In [8]:
def run_training(cfg: Config):
    seed_everything(cfg.seed)
    dev = get_device(cfg.device)
    ensure_dir(cfg.out_dir)
    torch.backends.cudnn.benchmark = True

    class_to_paths = discover_images(cfg.data_dir)
    class_to_imgs  = preload_images(class_to_paths)
    print("[data] classes & counts:", {k: len(v) for k, v in class_to_imgs.items()})
    print(f"[cfg] metric={cfg.metric} | use_attention={cfg.use_attention} | use_reliability={cfg.use_reliability}")

    backbone = ResNetBackbone(out_dim=cfg.embed_dim, use_pretrained=cfg.use_pretrained, use_attention=cfg.use_attention).to(dev)
    head = AdaptiveProtoHead(temperature=cfg.temperature, shrinkage=cfg.cov_shrinkage, metric=cfg.metric).to(dev)

    params = list(backbone.parameters()) + list(head.parameters())
    opt = optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)

    use_amp = (dev.type == "cuda")
    scaler = GradScaler("cuda") if use_amp else None
    amp_ctx = autocast("cuda") if use_amp else contextlib.nullcontext()

    best_val = 0.0
    for epoch in range(1, cfg.max_epochs+1):
        backbone.train(); head.train()
        losses, accs = [], []
        t0 = time.time()

        for epi in range(1, cfg.episodes_per_epoch + 1):
            supp_x, supp_y, qry_x, qry_y, _ = sample_episode(class_to_imgs, cfg.n_way, cfg.k_shot, cfg.q_queries, "train", cfg)
            supp_x, supp_y, qry_x, qry_y = supp_x.to(dev), supp_y.to(dev), qry_x.to(dev), qry_y.to(dev)

            with amp_ctx:
                s_emb = backbone(supp_x)
                q_emb = backbone(qry_x)
                reliability = backbone.quality_score(s_emb).detach() if cfg.use_reliability else torch.ones(s_emb.size(0), device=dev)
                logits, means = head(s_emb, supp_y, q_emb, cfg.n_way, reliability)
                loss = xent_loss(logits, qry_y) + 0.001 * proto_compactness_loss(s_emb, supp_y, means)

            opt.zero_grad(set_to_none=True)
            if use_amp:
                scaler.scale(loss).backward()
                if cfg.grad_clip is not None:
                    scaler.unscale_(opt); nn.utils.clip_grad_norm_(params, cfg.grad_clip)
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                if cfg.grad_clip is not None:
                    nn.utils.clip_grad_norm_(params, cfg.grad_clip)
                opt.step()

            preds = logits.argmax(dim=1)
            acc = (preds == qry_y).float().mean().item()
            accs.append(acc); losses.append(loss.item())

            if epi % 20 == 0:
                print(f"  [epoch {epoch:02d}] episode {epi:03d}/{cfg.episodes_per_epoch} "
                      f"loss {sum(losses)/len(losses):.4f} acc {sum(accs)/len(accs):.3f}")

        val_acc = evaluate(cfg, backbone, head, class_to_imgs, dev, n_episodes=cfg.val_episodes)
        if val_acc > best_val:
            best_val = val_acc
            torch.save({"backbone": backbone.state_dict(), "head": head.state_dict(), "cfg": cfg.__dict__},
                       os.path.join(cfg.out_dir, "best.pt"))

        print(f"[epoch {epoch:02d}] train loss {sum(losses)/len(losses):.4f} "
              f"train acc {sum(accs)/len(accs):.3f} | val acc {val_acc:.3f} "
              f"| time {time.time()-t0:.1f}s")

    print("Training complete. Best val acc:", best_val)
    return backbone, head

@torch.no_grad()
def evaluate(cfg: Config, backbone: nn.Module, head: nn.Module, class_to_imgs, device, n_episodes: int = 100):
    backbone.eval(); head.eval()
    accs = []
    use_amp = (device.type == "cuda")
    amp_ctx = autocast("cuda") if use_amp else contextlib.nullcontext()

    for _ in range(n_episodes):
        supp_x, supp_y, qry_x, qry_y, _ = sample_episode(class_to_imgs, cfg.n_way, cfg.k_shot, cfg.q_queries, "eval", cfg)
        supp_x, supp_y, qry_x, qry_y = supp_x.to(device), supp_y.to(device), qry_x.to(device), qry_y.to(device)
        with amp_ctx:
            s_emb = backbone(supp_x); q_emb = backbone(qry_x)
            reliability = backbone.quality_score(s_emb) if cfg.use_reliability else torch.ones(s_emb.size(0), device=device)
            logits, _ = head(s_emb, supp_y, q_emb, cfg.n_way, reliability)
        preds = logits.argmax(dim=1)
        acc = (preds == qry_y).float().mean().item()
        accs.append(acc)
    return sum(accs)/len(accs)

def load_best(cfg: Config, device=None):
    device = get_device(cfg.device) if device is None else device
    ckpt_path = os.path.join(cfg.out_dir, "best.pt")
    assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"
    backbone = ResNetBackbone(out_dim=cfg.embed_dim, use_pretrained=False, use_attention=cfg.use_attention).to(device)
    head = AdaptiveProtoHead(temperature=cfg.temperature, shrinkage=cfg.cov_shrinkage, metric=cfg.metric).to(device)
    state = torch.load(ckpt_path, map_location=device)
    backbone.load_state_dict(state["backbone"])
    head.load_state_dict(state["head"])
    return backbone, head


In [9]:
def pil_to_tensor(cfg: Config):
    return transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(cfg.normalize_mean, cfg.normalize_std),
    ])

@torch.no_grad()
def build_supports(class_to_imgs: Dict[str, List[Image.Image]], k_shot: int, backbone, device, cfg: Config):
    tform = pil_to_tensor(cfg)
    classes = sorted(class_to_imgs.keys())
    supp_x, supp_y = [], []
    for ci, cls in enumerate(classes):
        picks = class_to_imgs[cls][:k_shot]
        for img in picks:
            supp_x.append(tform(img))
            supp_y.append(ci)
    supp_x = torch.stack(supp_x, 0).to(device)
    supp_y = torch.tensor(supp_y, dtype=torch.long, device=device)
    s_emb = backbone(supp_x)
    return s_emb, supp_y, classes

@torch.no_grad()
def build_queries(class_to_imgs: Dict[str, List[Image.Image]], k_shot: int, cfg: Config, include_supports: bool = False):
    tform = pil_to_tensor(cfg)
    q_x, q_y, q_paths = [], [], []
    classes = sorted(class_to_imgs.keys())
    for ci, cls in enumerate(classes):
        imgs = class_to_imgs[cls] if include_supports else class_to_imgs[cls][k_shot:]
        for idx, img in enumerate(imgs):
            q_x.append(tform(img)); q_y.append(ci); q_paths.append(f"{cls}/img_{idx}.png")
    if len(q_x) == 0:
        raise RuntimeError("No query images assembled.")
    return torch.stack(q_x, 0), torch.tensor(q_y, dtype=torch.long), q_paths, classes

@torch.no_grad()
def run_fixed_eval(cfg: Config, k_shot: int = None, include_supports_in_queries: bool = False, batch_size: int = 128):
    device = get_device(cfg.device)
    if k_shot is None: k_shot = cfg.k_shot

    class_to_paths = discover_images(cfg.data_dir)
    class_to_imgs  = preload_images(class_to_paths)

    backbone, head = load_best(cfg, device)
    backbone.eval(); head.eval()

    s_emb, s_y, classes = build_supports(class_to_imgs, k_shot, backbone, device, cfg)
    reliability = backbone.quality_score(s_emb) if cfg.use_reliability else torch.ones(s_emb.size(0), device=device)
    q_x, q_y, q_paths, classes2 = build_queries(class_to_imgs, k_shot, cfg, include_supports=include_supports_in_queries)
    assert classes == classes2

    preds = []
    for start in range(0, q_x.size(0), batch_size):
        batch = q_x[start:start+batch_size].to(device)
        q_emb = backbone(batch)
        logits, _ = head(s_emb, s_y, q_emb, n_way=len(classes), reliability=reliability)
        preds.extend(logits.argmax(dim=1).cpu().tolist())

    y_true = q_y.cpu().tolist()
    y_pred = preds

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    os.makedirs(cfg.out_dir, exist_ok=True)
    np.savetxt(os.path.join(cfg.out_dir, "fixed_confusion.csv"), cm, fmt="%d", delimiter=",")
    with open(os.path.join(cfg.out_dir, "fixed_classes.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(classes))

    fig, ax = plt.subplots(figsize=(7,6))
    im = ax.imshow(cm)
    ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha="right"); ax.set_yticklabels(classes)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix (7-class)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

    report = classification_report(y_true, y_pred, labels=list(range(len(classes))), target_names=classes, digits=4)
    print(report)
    print(f"[saved] fixed_confusion.csv & fixed_classes.txt in {cfg.out_dir}")


In [10]:
def list_images_recursive(folder: str) -> List[str]:
    files = []
    for root, _, names in os.walk(folder):
        for fn in names:
            if os.path.splitext(fn)[1].lower() in IMG_EXTS:
                files.append(os.path.join(root, fn))
    return sorted(files)

@torch.no_grad()
def predict_folder(cfg: Config, folder: str, k_shot: int = None, save_csv: str = None, batch_size: int = 128):
    device = get_device(cfg.device)
    backbone, head = load_best(cfg, device)
    backbone.eval(); head.eval()

    class_to_imgs = preload_images(discover_images(cfg.data_dir))
    if k_shot is None: k_shot = cfg.k_shot

    tform = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(cfg.normalize_mean, cfg.normalize_std),
    ])

    # Build supports
    supp_x, supp_y, classes = [], [], []
    for ci, cls in enumerate(sorted(class_to_imgs.keys())):
        for img in class_to_imgs[cls][:k_shot]:
            supp_x.append(tform(img)); supp_y.append(ci)
        classes.append(cls)
    supp_x = torch.stack(supp_x, 0).to(device)
    supp_y = torch.tensor(supp_y, dtype=torch.long, device=device)
    s_emb = backbone(supp_x)
    reliability = backbone.quality_score(s_emb) if cfg.use_reliability else torch.ones(s_emb.size(0), device=device)

    # Queries
    image_paths = list_images_recursive(folder)
    if not image_paths:
        raise RuntimeError(f"No images found in: {folder}")

    results = []
    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start+batch_size]
        imgs, valid_idx = [], []
        for i, p in enumerate(batch_paths):
            try:
                imgs.append(tform(Image.open(p).convert("RGB"))); valid_idx.append(i)
            except Exception as e:
                print(f"[warn] skip: {p} ({e})")
        if not imgs: continue
        q_x = torch.stack(imgs, 0).to(device)
        q_emb = backbone(q_x)
        logits, _ = head(s_emb, supp_y, q_emb, n_way=len(classes), reliability=reliability)
        probs = torch.softmax(logits, dim=1)
        conf, pred = probs.max(dim=1)
        for i, cls_idx, c in zip(valid_idx, pred.cpu().tolist(), conf.cpu().tolist()):
            full = batch_paths[i]
            rel = os.path.relpath(full, start=folder)
            results.append((rel, classes[cls_idx], float(c)))

    if save_csv:
        dirn = os.path.dirname(save_csv)
        if dirn: os.makedirs(dirn, exist_ok=True)
        with open(save_csv, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f); w.writerow(["image", "pred_class", "confidence"]); w.writerows(results)
        print(f"[inference] wrote: {save_csv}")
    return results


In [13]:
import pandas as pd

def quick_ablation(cfg: Config, variants: List[Tuple[str, str, bool, bool]],
                   epochs=3, episodes_per_epoch=40, val_eps=20, test_eps=60):
    rows = []
    for name, metric, use_attn, use_rel in variants:
        cfg2 = Config(**cfg.__dict__)
        cfg2.metric = metric
        cfg2.use_attention = use_attn
        cfg2.use_reliability = use_rel
        cfg2.max_epochs = epochs
        cfg2.episodes_per_epoch = episodes_per_epoch
        cfg2.val_episodes = val_eps
        cfg2.test_episodes = test_eps
        cfg2.out_dir = os.path.join(cfg.out_dir, f"ablate_{name}")
        os.makedirs(cfg2.out_dir, exist_ok=True)
        t0 = time.time()
        backbone, head = run_training(cfg2)
        dur = time.time() - t0

        class_to_paths = discover_images(cfg2.data_dir)
        class_to_imgs  = preload_images(class_to_paths)
        acc = evaluate(cfg2, backbone, head, class_to_imgs, get_device(cfg2.device), n_episodes=cfg2.test_episodes)
        rows.append({"name": name, "metric": metric, "use_attention": use_attn, "use_reliability": use_rel,
                     "test_acc": acc, "train_seconds": dur})
    df = pd.DataFrame(rows).sort_values("test_acc", ascending=False).reset_index(drop=True)
    df
    return df

# define variants when you want to run (can be time-consuming):
variants = [
    ("maha_attn_rel",  "maha",  True,  True),
    ("maha_noattn_rel","maha",  False, True),
    ("maha_attn_norel","maha",  True,  False),
    ("euclid_attn_rel","euclid",True,  True),
]
print("Variants ready. Call quick_ablation(cfg, variants) to run.")


Variants ready. Call quick_ablation(cfg, variants) to run.


In [14]:
backbone, head = run_training(cfg)


[data] RAM-cached 280 images across 7 classes.
[data] classes & counts: {'BF': 40, 'BFI': 40, 'GF': 40, 'GFI': 40, 'N': 40, 'NI': 40, 'TF': 40}
[cfg] metric=maha | use_attention=True | use_reliability=True
  [epoch 01] episode 020/60 loss 41.1263 acc 0.254
  [epoch 01] episode 040/60 loss 29.7007 acc 0.304
  [epoch 01] episode 060/60 loss 22.5804 acc 0.329
[epoch 01] train loss 22.5804 train acc 0.329 | val acc 0.730 | time 281.6s
  [epoch 02] episode 020/60 loss 4.2412 acc 0.364
  [epoch 02] episode 040/60 loss 3.1185 acc 0.395
  [epoch 02] episode 060/60 loss 2.5511 acc 0.410
[epoch 02] train loss 2.5511 train acc 0.410 | val acc 0.700 | time 267.5s
  [epoch 03] episode 020/60 loss 1.2816 acc 0.404
  [epoch 03] episode 040/60 loss 1.1891 acc 0.409
  [epoch 03] episode 060/60 loss 1.1364 acc 0.418
[epoch 03] train loss 1.1364 train acc 0.418 | val acc 0.572 | time 273.9s
  [epoch 04] episode 020/60 loss 1.0152 acc 0.466
  [epoch 04] episode 040/60 loss 1.0143 acc 0.452
  [epoch 04] ep